In [3]:
import sys
sys.path.append("../")

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
from matplotlib import gridspec, ticker
from scipy.optimize import fsolve
import seaborn as sns
import numpy as np
from scipy.integrate import cumulative_trapezoid, quad
from tqdm import *
from scipy.interpolate import interp1d
import scipy.special as sp

# Dark photon perturbation theory modules
grf_path = "/home/bakerem/dark-photons-perturbations"
sys.path.append(grf_path)

from grf.grf import PerturbedProbability, FIRAS
from grf.pk_interp import PowerSpectrumGridInterpolator
from grf.units import *

from IPython.display import set_matplotlib_formats
set_matplotlib_formats('retina')

%matplotlib inline
%load_ext autoreload
%autoreload 2

# Custom plotting settings
# from plot_params import params
# pylab.rcParams.update(params)
# 
# cols_default = plt.rcParams['axes.prop_cycle'].by_key()['color']

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/tmp/ipykernel_28464/1846032536.py:25: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('retina')


In [4]:
# Initialize with linear baryon power spectrum
pspec_lin_baryon = PowerSpectrumGridInterpolator("lin_baryon")
prob = FIRAS(pspec_lin_baryon)

# Density fluctuation cutoff for lognormal model validity
one_plus_delta_bound = 1e2  # δ < 100 (conservative bound)

# Dark photon mass grids for different regimes
low_mA_list = np.geomspace(1.5e-14, 1e-13, 15)   # Low mass (recombination era)
halo_mA_list = np.geomspace(1e-13, 1e-11, 25)    # Halo mass range
mA_list = np.concatenate([halo_mA_list, low_mA_list]) * eV

# Extended mass range for comprehensive survey
mA_list = np.geomspace(1e-16, 1e-9, 300) * eV

In [ ]:
# Initialize arrays for total conversion probabilities
P_tot_an_ary = np.zeros_like(mA_list)

# Redshift array (exclude EoR to avoid 21cmFAST complications)
rs_array = np.geomspace(0.001, 17, 1000)
rs_array = rs_array[(rs_array < 5) | (rs_array > 35)]  # Skip 5 < z < 35

# Conversion probabilities with density fluctuation bound
P_tot_bounded_ary = np.zeros_like(mA_list)

print("Computing conversion probabilities for lognormal density field...")

for i_m_Ap, m_Ap in enumerate(tqdm(mA_list, desc="Dark photon masses")):
    # Physical parameters at z=17 (post-recombination reference)
    T0 = 2.73 * 8.62e-5  # CMB temperature in eV
    xobs = 0.0251/(1+17)  # Free electron fraction at z=17
    omega_ary = xobs * T0 * eV  # Plasma frequency
    
    # Compute dP/dz using lognormal PDF with density cutoff
    dPdrs_array = prob._dP_dz(
        z_ary=rs_array, 
        m_Ap=m_Ap, 
        k_min=1e-3, k_max=1e3,  # Integration bounds
        omega=omega_ary, 
        pdf='lognormal', 
        one_plus_delta_bound=one_plus_delta_bound
    )[0][0]
    
    # Integrate over redshift to get total probability
    P_tot_bounded_ary[i_m_Ap] = np.trapz(dPdrs_array, rs_array)

print(f"Conversion probabilities computed for {len(mA_list)} masses")

In [ ]:
def find_nearest(array, value):
    """Find nearest array element to target value."""
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]

# Observational constraint: global signal deviation limit
bt_limit = -100  # mK (conservative bound)
z = 18  # Reference redshift (post-recombination)
Tgamma0_mK = 2.73 * 1000  # CMB monopole in mK

# Find reference redshift in array
z_index = np.where(rs_array == find_nearest(rs_array, z))[0]
bt_at_z = 0  # Fiducial ΛCDM brightness temperature
xobs = 0.0251/(1+z)  # Free electron fraction
xe = prob.x_e(rs_array)  # Ionization history

print("Computing coupling limits using lognormal conversion probabilities...")

# Solve for coupling limit at each mass
lognorm_lims = []
signal_ary = []
for i_m_Ap, m_Ap in enumerate(mA_list):
    signal_ary.append(-Tgamma0_mK * P_tot_bounded_ary[i_m_Ap])
